# X1: Deployment and Production RL Systems

**Learning Objectives:**
- Deploy RL models to production environments
- Implement model serving infrastructure
- Handle online vs offline RL in production
- Build scalable training pipelines
- Implement safety constraints and guardrails
- Monitor deployed RL systems
- Handle distribution shift in production

**Prerequisites:** All main curriculum (Lessons 0-16)

**Real-World Examples:**
- Google Ads bidding systems
- YouTube recommendation algorithms
- Tesla Autopilot
- OpenAI ChatGPT API
- Robotics deployment

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import deque
import json
import time
from pathlib import Path
import pickle
from typing import Dict, Any, Optional

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Model Serialization and Versioning

Production RL requires robust model management.

In [ ]:
class RLModelManager:
    """Manages RL model serialization, versioning, and loading."""
    
    def __init__(self, model_dir="./models"):
        self.model_dir = Path(model_dir)
        self.model_dir.mkdir(parents=True, exist_ok=True)
    
    def save_model(self, model, metadata: Dict[str, Any], version: str):
        """Save model with metadata and version.
        
        Args:
            model: PyTorch model
            metadata: Dict with training info, hyperparameters, etc.
            version: Version string (e.g., 'v1.0.0', 'exp_20250123')
        """
        version_dir = self.model_dir / version
        version_dir.mkdir(parents=True, exist_ok=True)
        
        # Save model weights
        model_path = version_dir / "model.pt"
        torch.save(model.state_dict(), model_path)
        
        # Save complete model (for architecture)
        full_model_path = version_dir / "model_full.pt"
        torch.save(model, full_model_path)
        
        # Save metadata
        metadata_path = version_dir / "metadata.json"
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✓ Model saved to {version_dir}")
        return version_dir
    
    def load_model(self, version: str, architecture=None):
        """Load model by version.
        
        Args:
            version: Version string
            architecture: Optional model architecture (if loading state_dict)
        
        Returns:
            model: Loaded PyTorch model
            metadata: Model metadata
        """
        version_dir = self.model_dir / version
        
        if not version_dir.exists():
            raise ValueError(f"Version {version} not found")
        
        # Load metadata
        metadata_path = version_dir / "metadata.json"
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
        
        # Load model
        if architecture is not None:
            # Load state dict into provided architecture
            model_path = version_dir / "model.pt"
            architecture.load_state_dict(torch.load(model_path))
            model = architecture
        else:
            # Load full model
            full_model_path = version_dir / "model_full.pt"
            model = torch.load(full_model_path)
        
        model.eval()
        print(f"✓ Model loaded from {version_dir}")
        return model, metadata
    
    def list_versions(self):
        """List all available model versions."""
        versions = [d.name for d in self.model_dir.iterdir() if d.is_dir()]
        return sorted(versions)

# Example usage
manager = RLModelManager("./rl_models")

# Create example policy
class SimplePolicy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Softmax(dim=-1)
        )
    
    def forward(self, x):
        return self.net(x)

policy = SimplePolicy(state_dim=4, action_dim=2)

# Save with metadata
metadata = {
    "algorithm": "PPO",
    "environment": "CartPole-v1",
    "training_steps": 100000,
    "avg_reward": 475.3,
    "hyperparameters": {
        "lr": 3e-4,
        "gamma": 0.99,
        "clip_epsilon": 0.2
    },
    "timestamp": "2025-01-23T10:00:00Z"
}

manager.save_model(policy, metadata, version="v1.0.0")

print(f"\nAvailable versions: {manager.list_versions()}")

print("\nModel management implemented")

## 2. Model Serving Infrastructure

Fast, reliable inference for production.

In [ ]:
class RLModelServer:
    """Production model server for RL inference."""
    
    def __init__(self, model, preprocessing_fn=None, postprocessing_fn=None):
        """
        Args:
            model: PyTorch model
            preprocessing_fn: Optional function to preprocess inputs
            postprocessing_fn: Optional function to postprocess outputs
        """
        self.model = model.to(device)
        self.model.eval()
        self.preprocessing_fn = preprocessing_fn or (lambda x: x)
        self.postprocessing_fn = postprocessing_fn or (lambda x: x)
        
        # Performance tracking
        self.request_count = 0
        self.total_latency = 0.0
        self.errors = 0
        
        # Warm up
        self._warmup()
    
    def _warmup(self):
        """Warm up model with dummy inference."""
        print("Warming up model...")
        # Get input shape from model
        if hasattr(self.model, 'net'):
            first_layer = list(self.model.net.children())[0]
            if hasattr(first_layer, 'in_features'):
                input_dim = first_layer.in_features
                dummy_input = torch.randn(1, input_dim).to(device)
                with torch.no_grad():
                    _ = self.model(dummy_input)
                print("✓ Model warmed up")
    
    def predict(self, state, deterministic=True):
        """Get action for state.
        
        Args:
            state: Environment state
            deterministic: If True, return argmax; else sample
        
        Returns:
            action: Selected action
            info: Dict with probabilities, latency, etc.
        """
        start_time = time.time()
        
        try:
            # Preprocess
            state = self.preprocessing_fn(state)
            
            # Convert to tensor
            if not isinstance(state, torch.Tensor):
                state = torch.FloatTensor(state).to(device)
            
            if state.dim() == 1:
                state = state.unsqueeze(0)
            
            # Inference
            with torch.no_grad():
                action_probs = self.model(state)
            
            # Select action
            if deterministic:
                action = action_probs.argmax(dim=-1).item()
            else:
                dist = torch.distributions.Categorical(action_probs)
                action = dist.sample().item()
            
            # Postprocess
            action = self.postprocessing_fn(action)
            
            latency = time.time() - start_time
            
            # Track metrics
            self.request_count += 1
            self.total_latency += latency
            
            info = {
                'probabilities': action_probs.cpu().numpy().tolist(),
                'latency_ms': latency * 1000,
                'deterministic': deterministic
            }
            
            return action, info
        
        except Exception as e:
            self.errors += 1
            raise RuntimeError(f"Inference error: {e}")
    
    def batch_predict(self, states, deterministic=True):
        """Batched inference for efficiency."""
        start_time = time.time()
        
        # Preprocess batch
        states = [self.preprocessing_fn(s) for s in states]
        states_tensor = torch.FloatTensor(states).to(device)
        
        # Inference
        with torch.no_grad():
            action_probs = self.model(states_tensor)
        
        # Select actions
        if deterministic:
            actions = action_probs.argmax(dim=-1).cpu().numpy()
        else:
            dist = torch.distributions.Categorical(action_probs)
            actions = dist.sample().cpu().numpy()
        
        # Postprocess
        actions = [self.postprocessing_fn(a) for a in actions]
        
        latency = time.time() - start_time
        self.request_count += len(states)
        self.total_latency += latency
        
        return actions, {'batch_latency_ms': latency * 1000}
    
    def get_stats(self):
        """Get server statistics."""
        avg_latency = (self.total_latency / self.request_count * 1000 
                      if self.request_count > 0 else 0)
        
        return {
            'total_requests': self.request_count,
            'avg_latency_ms': avg_latency,
            'errors': self.errors,
            'error_rate': self.errors / max(self.request_count, 1)
        }

# Example usage
server = RLModelServer(policy)

# Single prediction
state = np.array([0.1, 0.2, 0.3, 0.4])
action, info = server.predict(state, deterministic=True)
print(f"\nSingle prediction:")
print(f"  Action: {action}")
print(f"  Latency: {info['latency_ms']:.2f} ms")

# Batch prediction
states = [np.random.randn(4) for _ in range(100)]
actions, batch_info = server.batch_predict(states)
print(f"\nBatch prediction (100 states):")
print(f"  Latency: {batch_info['batch_latency_ms']:.2f} ms")
print(f"  Per-sample: {batch_info['batch_latency_ms']/100:.2f} ms")

# Server stats
stats = server.get_stats()
print(f"\nServer statistics:")
print(f"  Total requests: {stats['total_requests']}")
print(f"  Avg latency: {stats['avg_latency_ms']:.2f} ms")
print(f"  Error rate: {stats['error_rate']:.4f}")

print("\nModel serving implemented")

## 3. Online Learning in Production

Continual learning from live data.

In [ ]:
class OnlineRLSystem:
    """Online RL with experience collection and periodic updates."""
    
    def __init__(self, policy, optimizer, buffer_size=10000, 
                 update_freq=1000, batch_size=64):
        self.policy = policy
        self.optimizer = optimizer
        self.buffer = deque(maxlen=buffer_size)
        self.update_freq = update_freq
        self.batch_size = batch_size
        
        self.steps = 0
        self.updates = 0
        self.update_history = []
    
    def collect_experience(self, state, action, reward, next_state, done):
        """Add experience to buffer."""
        self.buffer.append((state, action, reward, next_state, done))
        self.steps += 1
        
        # Periodic update
        if self.steps % self.update_freq == 0 and len(self.buffer) >= self.batch_size:
            loss = self.update_policy()
            return loss
        return None
    
    def update_policy(self):
        """Update policy from buffer (simplified PPO-style)."""
        # Sample batch
        indices = np.random.choice(len(self.buffer), self.batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        
        states = torch.FloatTensor([b[0] for b in batch]).to(device)
        actions = torch.LongTensor([b[1] for b in batch]).to(device)
        rewards = torch.FloatTensor([b[2] for b in batch]).to(device)
        
        # Simplified policy gradient
        action_probs = self.policy(states)
        log_probs = torch.log(action_probs.gather(1, actions.unsqueeze(1)).squeeze())
        loss = -(log_probs * rewards).mean()
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.updates += 1
        self.update_history.append(loss.item())
        
        return loss.item()
    
    def get_diagnostics(self):
        """Get system diagnostics."""
        return {
            'steps': self.steps,
            'updates': self.updates,
            'buffer_size': len(self.buffer),
            'recent_loss': self.update_history[-1] if self.update_history else None
        }

# Example: Simulate online learning
online_policy = SimplePolicy(4, 2)
optimizer = torch.optim.Adam(online_policy.parameters(), lr=1e-4)
online_system = OnlineRLSystem(online_policy, optimizer, update_freq=500)

print("Simulating online learning...")
for step in range(5000):
    # Simulate experience
    state = np.random.randn(4)
    action = np.random.randint(0, 2)
    reward = np.random.randn()
    next_state = np.random.randn(4)
    done = False
    
    loss = online_system.collect_experience(state, action, reward, next_state, done)
    
    if loss is not None and step % 1000 == 0:
        print(f"  Step {step}, Loss: {loss:.4f}")

diag = online_system.get_diagnostics()
print(f"\nOnline system diagnostics:")
print(f"  Total steps: {diag['steps']}")
print(f"  Policy updates: {diag['updates']}")
print(f"  Buffer size: {diag['buffer_size']}")

print("\nOnline learning system implemented")

## 4. Safety Constraints and Guardrails

Critical for production deployment.

In [ ]:
class SafeRLWrapper:
    """Wrapper that enforces safety constraints on RL policy."""
    
    def __init__(self, policy, constraints=None, fallback_policy=None):
        """
        Args:
            policy: RL policy
            constraints: List of constraint functions (state, action) -> bool
            fallback_policy: Safe fallback policy if constraints violated
        """
        self.policy = policy
        self.constraints = constraints or []
        self.fallback_policy = fallback_policy
        
        # Safety monitoring
        self.total_actions = 0
        self.violations = 0
        self.fallback_activations = 0
    
    def add_constraint(self, constraint_fn):
        """Add safety constraint."""
        self.constraints.append(constraint_fn)
    
    def check_constraints(self, state, action):
        """Check if action violates any constraint."""
        for constraint in self.constraints:
            if not constraint(state, action):
                return False
        return True
    
    def get_safe_action(self, state, max_attempts=5):
        """Get action that satisfies constraints."""
        self.total_actions += 1
        
        # Try policy action
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        for attempt in range(max_attempts):
            with torch.no_grad():
                probs = self.policy(state_tensor)
            
            # Sample action
            dist = torch.distributions.Categorical(probs)
            action = dist.sample().item()
            
            # Check constraints
            if self.check_constraints(state, action):
                return action, {'safe': True, 'attempts': attempt + 1}
            
            self.violations += 1
        
        # Fallback if all attempts fail
        self.fallback_activations += 1
        
        if self.fallback_policy is not None:
            action = self.fallback_policy(state)
            return action, {'safe': True, 'fallback': True}
        else:
            # Default safe action (e.g., action 0)
            return 0, {'safe': True, 'fallback': True, 'default': True}
    
    def get_safety_stats(self):
        """Get safety statistics."""
        return {
            'total_actions': self.total_actions,
            'violations': self.violations,
            'fallback_activations': self.fallback_activations,
            'violation_rate': self.violations / max(self.total_actions, 1),
            'fallback_rate': self.fallback_activations / max(self.total_actions, 1)
        }

# Example constraints
def state_bound_constraint(state, action):
    """Constraint: state values must be within bounds."""
    return np.all(np.abs(state) < 10.0)

def action_consistency_constraint(state, action):
    """Constraint: action should be consistent with state sign."""
    # Example: if state[0] > 0, prefer action 1
    # This is a toy example
    return True  # Always allow for demo

# Fallback policy
def safe_fallback(state):
    """Conservative fallback policy."""
    return 0  # Always choose action 0 (safe default)

# Create safe wrapper
safe_policy = SafeRLWrapper(
    policy,
    constraints=[state_bound_constraint, action_consistency_constraint],
    fallback_policy=safe_fallback
)

# Test safety wrapper
print("Testing safety wrapper...")
for _ in range(1000):
    state = np.random.randn(4) * 5  # Random states
    action, info = safe_policy.get_safe_action(state)

stats = safe_policy.get_safety_stats()
print(f"\nSafety statistics:")
print(f"  Total actions: {stats['total_actions']}")
print(f"  Violations: {stats['violations']}")
print(f"  Fallback activations: {stats['fallback_activations']}")
print(f"  Violation rate: {stats['violation_rate']:.4f}")

print("\nSafety wrapper implemented")

## 5. A/B Testing for RL Policies

Safe deployment via gradual rollout.

In [ ]:
class ABTestingFramework:
    """A/B testing framework for RL policies."""
    
    def __init__(self, baseline_policy, candidate_policy, rollout_percentage=10):
        """
        Args:
            baseline_policy: Current production policy
            candidate_policy: New policy to test
            rollout_percentage: % of traffic to send to candidate
        """
        self.baseline = baseline_policy
        self.candidate = candidate_policy
        self.rollout_pct = rollout_percentage
        
        # Metrics
        self.baseline_rewards = []
        self.candidate_rewards = []
        self.baseline_count = 0
        self.candidate_count = 0
    
    def select_policy(self, user_id=None):
        """Select policy based on rollout percentage."""
        # Hash-based assignment for consistency per user
        if user_id is not None:
            hash_val = hash(user_id) % 100
        else:
            hash_val = np.random.randint(0, 100)
        
        if hash_val < self.rollout_pct:
            return 'candidate', self.candidate
        else:
            return 'baseline', self.baseline
    
    def log_result(self, policy_name, reward):
        """Log result from policy."""
        if policy_name == 'candidate':
            self.candidate_rewards.append(reward)
            self.candidate_count += 1
        else:
            self.baseline_rewards.append(reward)
            self.baseline_count += 1
    
    def get_results(self):
        """Get A/B test results."""
        baseline_mean = np.mean(self.baseline_rewards) if self.baseline_rewards else 0
        candidate_mean = np.mean(self.candidate_rewards) if self.candidate_rewards else 0
        
        baseline_std = np.std(self.baseline_rewards) if self.baseline_rewards else 0
        candidate_std = np.std(self.candidate_rewards) if self.candidate_rewards else 0
        
        # Simple significance test (t-test approximation)
        if len(self.baseline_rewards) > 30 and len(self.candidate_rewards) > 30:
            pooled_std = np.sqrt(
                (baseline_std**2 / len(self.baseline_rewards)) +
                (candidate_std**2 / len(self.candidate_rewards))
            )
            if pooled_std > 0:
                t_stat = (candidate_mean - baseline_mean) / pooled_std
                significant = abs(t_stat) > 1.96  # 95% confidence
            else:
                significant = False
        else:
            significant = False
        
        improvement = ((candidate_mean - baseline_mean) / abs(baseline_mean) * 100 
                      if baseline_mean != 0 else 0)
        
        return {
            'baseline': {
                'mean': baseline_mean,
                'std': baseline_std,
                'count': self.baseline_count
            },
            'candidate': {
                'mean': candidate_mean,
                'std': candidate_std,
                'count': self.candidate_count
            },
            'improvement_pct': improvement,
            'statistically_significant': significant
        }
    
    def should_promote_candidate(self, min_improvement=5.0, min_samples=100):
        """Decide if candidate should be promoted to production."""
        results = self.get_results()
        
        # Criteria for promotion
        enough_samples = (results['baseline']['count'] >= min_samples and 
                         results['candidate']['count'] >= min_samples)
        significant = results['statistically_significant']
        improved = results['improvement_pct'] >= min_improvement
        
        return enough_samples and significant and improved

# Example A/B test
print("Running A/B test...")

baseline_policy = SimplePolicy(4, 2)
candidate_policy = SimplePolicy(4, 2)  # In practice, this would be different

ab_test = ABTestingFramework(baseline_policy, candidate_policy, rollout_percentage=20)

# Simulate episodes
for episode in range(500):
    user_id = f"user_{episode % 100}"  # Simulate 100 users
    policy_name, policy = ab_test.select_policy(user_id)
    
    # Simulate episode reward (candidate slightly better)
    if policy_name == 'candidate':
        reward = np.random.normal(10.5, 2.0)
    else:
        reward = np.random.normal(10.0, 2.0)
    
    ab_test.log_result(policy_name, reward)

# Get results
results = ab_test.get_results()
print(f"\nA/B Test Results:")
print(f"\nBaseline:")
print(f"  Mean reward: {results['baseline']['mean']:.2f}")
print(f"  Std: {results['baseline']['std']:.2f}")
print(f"  Samples: {results['baseline']['count']}")
print(f"\nCandidate:")
print(f"  Mean reward: {results['candidate']['mean']:.2f}")
print(f"  Std: {results['candidate']['std']:.2f}")
print(f"  Samples: {results['candidate']['count']}")
print(f"\nImprovement: {results['improvement_pct']:.2f}%")
print(f"Statistically significant: {results['statistically_significant']}")

promote = ab_test.should_promote_candidate(min_improvement=2.0)
print(f"\nRecommendation: {'PROMOTE candidate' if promote else 'Keep baseline'}")

print("\nA/B testing framework implemented")

## 6. Monitoring and Alerting

Track production system health.

In [ ]:
class RLMonitor:
    """Monitor RL system metrics and alert on anomalies."""
    
    def __init__(self, alert_thresholds=None):
        self.metrics = {
            'rewards': deque(maxlen=1000),
            'latencies': deque(maxlen=1000),
            'errors': deque(maxlen=1000),
            'action_distribution': {}
        }
        
        self.alert_thresholds = alert_thresholds or {
            'reward_drop_pct': 20,  # Alert if reward drops >20%
            'latency_p95_ms': 100,  # Alert if p95 latency >100ms
            'error_rate': 0.05      # Alert if error rate >5%
        }
        
        self.alerts = []
    
    def log_episode(self, reward, latency, action, error=False):
        """Log episode metrics."""
        self.metrics['rewards'].append(reward)
        self.metrics['latencies'].append(latency)
        self.metrics['errors'].append(1 if error else 0)
        
        # Track action distribution
        self.metrics['action_distribution'][action] = \
            self.metrics['action_distribution'].get(action, 0) + 1
    
    def check_alerts(self):
        """Check for alert conditions."""
        alerts = []
        
        if len(self.metrics['rewards']) >= 100:
            # Reward drop
            recent_reward = np.mean(list(self.metrics['rewards'])[-100:])
            baseline_reward = np.mean(list(self.metrics['rewards'])[:100])
            
            if baseline_reward > 0:
                drop_pct = (baseline_reward - recent_reward) / baseline_reward * 100
                if drop_pct > self.alert_thresholds['reward_drop_pct']:
                    alerts.append({
                        'type': 'REWARD_DROP',
                        'severity': 'HIGH',
                        'message': f'Reward dropped {drop_pct:.1f}%',
                        'baseline': baseline_reward,
                        'current': recent_reward
                    })
            
            # Latency
            p95_latency = np.percentile(self.metrics['latencies'], 95)
            if p95_latency > self.alert_thresholds['latency_p95_ms']:
                alerts.append({
                    'type': 'HIGH_LATENCY',
                    'severity': 'MEDIUM',
                    'message': f'P95 latency: {p95_latency:.1f}ms',
                    'threshold': self.alert_thresholds['latency_p95_ms']
                })
            
            # Error rate
            error_rate = np.mean(self.metrics['errors'])
            if error_rate > self.alert_thresholds['error_rate']:
                alerts.append({
                    'type': 'HIGH_ERROR_RATE',
                    'severity': 'CRITICAL',
                    'message': f'Error rate: {error_rate*100:.2f}%',
                    'threshold': self.alert_thresholds['error_rate'] * 100
                })
        
        self.alerts.extend(alerts)
        return alerts
    
    def get_dashboard(self):
        """Get monitoring dashboard metrics."""
        if not self.metrics['rewards']:
            return {}
        
        return {
            'reward': {
                'mean': np.mean(self.metrics['rewards']),
                'p50': np.percentile(self.metrics['rewards'], 50),
                'p95': np.percentile(self.metrics['rewards'], 95)
            },
            'latency': {
                'mean': np.mean(self.metrics['latencies']),
                'p50': np.percentile(self.metrics['latencies'], 50),
                'p95': np.percentile(self.metrics['latencies'], 95),
                'p99': np.percentile(self.metrics['latencies'], 99)
            },
            'error_rate': np.mean(self.metrics['errors']) * 100,
            'action_distribution': dict(self.metrics['action_distribution']),
            'total_episodes': len(self.metrics['rewards']),
            'active_alerts': len([a for a in self.alerts 
                                 if a.get('severity') in ['HIGH', 'CRITICAL']])
        }

# Example monitoring
monitor = RLMonitor()

print("Simulating production episodes...")

# Normal operation
for i in range(500):
    reward = np.random.normal(10, 1)
    latency = np.random.gamma(2, 5)  # ms
    action = np.random.choice([0, 1], p=[0.6, 0.4])
    error = np.random.random() < 0.01
    
    monitor.log_episode(reward, latency, action, error)

# Degrade performance
print("  Introducing performance degradation...")
for i in range(200):
    reward = np.random.normal(7, 2)  # Worse rewards
    latency = np.random.gamma(4, 10)  # Higher latency
    action = np.random.choice([0, 1])
    error = np.random.random() < 0.08  # More errors
    
    monitor.log_episode(reward, latency, action, error)

# Check alerts
alerts = monitor.check_alerts()

# Dashboard
dashboard = monitor.get_dashboard()
print(f"\nMonitoring Dashboard:")
print(f"\nReward:")
print(f"  Mean: {dashboard['reward']['mean']:.2f}")
print(f"  P50: {dashboard['reward']['p50']:.2f}")
print(f"  P95: {dashboard['reward']['p95']:.2f}")
print(f"\nLatency:")
print(f"  Mean: {dashboard['latency']['mean']:.2f} ms")
print(f"  P95: {dashboard['latency']['p95']:.2f} ms")
print(f"  P99: {dashboard['latency']['p99']:.2f} ms")
print(f"\nError rate: {dashboard['error_rate']:.2f}%")
print(f"Total episodes: {dashboard['total_episodes']}")
print(f"Active alerts: {dashboard['active_alerts']}")

if alerts:
    print(f"\n⚠️  Alerts ({len(alerts)}):")
    for alert in alerts:
        print(f"  [{alert['severity']}] {alert['type']}: {alert['message']}")

print("\nMonitoring system implemented")

## Summary

### Production RL Components:

1. **Model Management**
   - Versioning and serialization
   - Metadata tracking
   - Reproducibility

2. **Model Serving**
   - Fast inference (< 100ms)
   - Batch processing
   - Warm-up and optimization

3. **Online Learning**
   - Experience collection
   - Periodic updates
   - Buffer management

4. **Safety**
   - Constraint checking
   - Fallback policies
   - Violation tracking

5. **A/B Testing**
   - Gradual rollout
   - Statistical significance
   - Automated decisions

6. **Monitoring**
   - Metrics tracking
   - Alert conditions
   - Dashboard visualization

### Best Practices:

✅ **Version everything**: Models, data, code
✅ **Monitor continuously**: Rewards, latency, errors
✅ **Deploy gradually**: A/B test before full rollout
✅ **Enforce safety**: Constraints and fallbacks
✅ **Track distribution shift**: Compare production vs training
✅ **Automate rollbacks**: Revert on performance drops

### Common Pitfalls:

❌ Deploying without monitoring
❌ No safety constraints
❌ Ignoring distribution shift
❌ Not versioning models
❌ Full rollout without testing

### Next:

**X2**: Hyperparameter tuning at scale  
**X3**: Debugging and troubleshooting  
**X4**: Real-world case studies